# 모던워페어4는 정말 더 관심을 받았을까? (재현 노트북)

DataPopcorn 테크블로그 글 [datapopcorn.ai/blog/modern-warfare-4-korea-search-data](https://datapopcorn.ai/blog/modern-warfare-4-korea-search-data)의 재현 노트북입니다. `런타임 → 모두 실행`을 누르면 글에 나온 숫자와 차트 4개가 다시 만들어집니다.

- 데이터: 2026-10-24에 수집한 스냅샷(네이버 데이터랩 주간·일간·연령·성별, 네이버 검색광고 키워드도구 최근 30일, Steam 현재 동시접속)
- API 키 없이 실행됩니다. 직접 다시 수집하려면 맨 아래 **부록: 수집 코드**를 보세요.

In [ ]:
# 0. 준비: 한글 폰트 + 데이터 내려받기 (로컬에서는 DATA_DIR 환경변수로 폴더 지정)
import os, sys, urllib.request, urllib.parse
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    os.system('apt-get -qq install -y fonts-nanum > /dev/null')
for f in ['/usr/share/fonts/truetype/nanum/NanumGothic.ttf', '/System/Library/Fonts/AppleSDGothicNeo.ttc', 'C:/Windows/Fonts/malgun.ttf']:
    if os.path.exists(f):
        os.environ['KFONT'] = f; break
FILES = ['week_titles.json', 'daily_mw4.json', 'week_age_mw4_mw2.json', 'week_gender_f.json', 'week_gender_m.json', 'searchad_monthly.txt', 'steam_current_players.json']
BASE = os.environ.get('DATA_DIR', 'https://raw.githubusercontent.com/team-datapopcorn/datapopcorn-assets/main/modern-warfare-4-korea-search-data/data')
if BASE.startswith('http'):
    os.makedirs('data', exist_ok=True)
    for fn in FILES:
        urllib.request.urlretrieve(f'{BASE}/{urllib.parse.quote(fn)}', f'data/{fn}')
    RAW = 'data'
else:
    RAW = BASE
os.environ['RAW'] = RAW; os.environ['FIG'] = 'fig'; os.environ['OUT'] = 'summary.json'
print('데이터 폴더:', RAW, sorted(os.listdir(RAW)))

In [ ]:
import json,re,os,pandas as pd,numpy as np,matplotlib
matplotlib.use('Agg');import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
FONT=os.environ.get('KFONT');fm.fontManager.addfont(FONT)
plt.rcParams.update({'font.family':fm.FontProperties(fname=FONT).get_name(),'axes.unicode_minus':False,'figure.dpi':150,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
C={'main':'#E8590C','sub':'#1C7ED6','gray':'#868E96','light':'#CED4DA'}
RAW=os.environ.get('RAW','raw');FIG=os.environ.get('FIG','fig');os.makedirs(FIG,exist_ok=True);S={}
def dl(f,key=None):
    j=json.load(open(f'{RAW}/{f}'));j=j if key is None else j['byAge'][key]
    return pd.DataFrame({r['title']:pd.Series({pd.Timestamp(x['period']):x['ratio'] for x in r['data']}) for r in j['results']}).fillna(0)

## ① 역대작 출시 주 비교 (Step 3)
다섯 작품을 한 요청에 넣고, 작품마다 같은 형태의 표기 6~7개를 묶었습니다(`week_titles.json`의 `groups`).

In [ ]:
# ---------- 1. 역대작 출시 주 비교 ----------
w=dl('week_titles.json');w=w.reindex(pd.date_range('2022-01-03',w.index.max(),freq='W-MON')).fillna(0)
order=[('모던워페어2','모던워페어2\n(2022)'),('모던워페어3','모던워페어3\n(2023)'),('블랙옵스6','블랙옵스6\n(2024)'),('블랙옵스7','블랙옵스7\n(2025)'),('모던워페어4','모던워페어4\n(2026)')]
pk={k:float(w[k].max()) for k,_ in order};pkw={k:str(w[k].idxmax().date()) for k,_ in order}
S['launch_week_peak']=pk;S['peak_week']=pkw;S['mw4_over']={k:round(100/v,1) for k,v in pk.items() if k!='모던워페어4'}
fig,ax=plt.subplots(figsize=(10,4.4));vals=[pk[k] for k,_ in order]
ax.bar([l for _,l in order],vals,color=[C['light']]*4+[C['main']])
for i,v in enumerate(vals):ax.text(i,v+1.5,f'{v:.1f}',ha='center')
ax.set_ylabel('주간 검색지수 최고값\n(같은 요청, 최고 주=100)')
ax.set_title(f"① 출시 주 네이버 검색, 모던워페어4가 작년 블랙옵스7의 {S['mw4_over']['블랙옵스7']:.0f}배·2022년 모던워페어2의 {S['mw4_over']['모던워페어2']}배\n각 작품 이름 검색의 주간 최고값 (2022~2026)",loc='left')
fig.text(0.01,-0.03,'자료: 네이버 데이터랩 검색어트렌드(주간, 5개 작품을 한 요청으로, 작품마다 같은 형태의 표기 6~7개를 묶음). 모던워페어4는 10/19~23 닷새만 집계된 주. 2026-10-24 조회',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig1_titles.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig1_titles.png'))

## ② 일별 급변점 (Step 4)
모던워페어4·콜오브듀티·배틀필드6을 한 요청으로 받은 일간 지수입니다. 값이 없는 날은 0으로 채웁니다.

In [ ]:
# ---------- 2. 일별 흐름 ----------
d=dl('daily_mw4.json').reindex(pd.date_range('2026-05-01','2026-10-23')).fillna(0)
S['daily_reveal_peak']={'date':str(d.loc[:'2026-09-30','모던워페어4'].idxmax().date()),'v':round(d.loc[:'2026-09-30','모던워페어4'].max(),1)}
S['daily_1015_1023']={str(k.date()):round(v,1) for k,v in d.loc['2026-10-14':'2026-10-23','모던워페어4'].items()}
fig,ax=plt.subplots(figsize=(12,4.4))
ax.plot(d.index,d['모던워페어4'],color=C['main'],lw=2,label="'모던워페어4'");ax.plot(d.index,d['콜오브듀티'],color=C['sub'],lw=1.4,label="'콜오브듀티'");ax.plot(d.index,d['배틀필드6'],color=C['gray'],lw=1,label="'배틀필드6'")
for x,t,y in [('2026-05-29','5/29 공개 트레일러 무렵',42),('2026-10-16','10/16 캠페인 사전 체험',62),('2026-10-23','10/23 정식 출시',104)]:
    ax.annotate(t,xy=(pd.Timestamp(x),min(y-4,100)),xytext=(pd.Timestamp(x)-pd.Timedelta(days=38 if x>'2026-10' else -4),y),fontsize=9,arrowprops=dict(arrowstyle='->',color=C['gray']))
ax.set_ylabel('검색지수 (기간 최고일=100)');ax.legend(frameon=False,loc='upper left')
ax.set_title('② 공개 때 한 번, 사전 체험 다음 날부터 매일 올라 출시일에 정점\n네이버 데이터랩 일별 검색지수 (2026.5~10)',loc='left')
fig.text(0.01,-0.03,'자료: 네이버 데이터랩(일간, 한 요청 안에서 비교). 일정은 콜 오브 듀티 공식 블로그 공지 기준. 2026-10-24 조회',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig2_daily.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig2_daily.png'))

## ③ 누가 늘었나 (Step 5)
연령·성별 지수는 집단마다 따로 100이므로, 각 집단 안에서 모던워페어4 ÷ 모던워페어2 최고값 비율로만 비교합니다.

In [ ]:
# ---------- 3. 누가 늘었나 ----------
A={}
for a in ['10대','20대','30대','40대','50대+']:
    x=dl('week_age_mw4_mw2.json',a);A[a]=round(x['모던워페어4'].max()/x['모던워페어2'].max(),1)
G={}
for g,lab in [('f','여성'),('m','남성')]:
    x=dl(f'week_gender_{g}.json');G[lab]=round(x['모던워페어4'].max()/x['모던워페어2'].max(),1)
S['mw4_over_mw2_by_age']=A;S['mw4_over_mw2_by_gender']=G
fig,ax=plt.subplots(1,2,figsize=(12,4),gridspec_kw={'width_ratios':[5,2]})
ax[0].bar(list(A),list(A.values()),color=[C['light'],C['light'],C['main'],C['main'],C['main']]);[ax[0].text(i,v+0.2,f'{v}배',ha='center') for i,v in enumerate(A.values())]
ax[1].bar(list(G),list(G.values()),color=[C['main'],C['light']]);[ax[1].text(i,v+0.2,f'{v}배',ha='center') for i,v in enumerate(G.values())]
ax[0].set_ylabel('모던워페어4 ÷ 모던워페어2\n(출시 주 최고값 비율)')
fig.suptitle('③ 전작보다 크게 늘어난 쪽은 10·20대가 아니라 30대 이상과 여성',x=0.01,ha='left',fontsize=12)
fig.text(0.01,-0.04,'자료: 네이버 데이터랩(연령·성별 필터, 각 집단 안에서 두 작품 비율). 집단끼리 검색량 크기를 비교한 값이 아님',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig3_who.png',bbox_inches='tight');plt.close()
from IPython.display import Image, display
display(Image(f'{FIG}/fig3_who.png'))

## ④ 최근 30일 실제 검색수 (Step 2)
키워드도구의 최근 30일 PC+모바일 검색수입니다. 띄어쓰기 없는 표기로 집계됩니다.

In [ ]:
# ---------- 4. 최근 30일 검색수 ----------
sa={}
for line in open(f'{RAW}/searchad_monthly.txt'):
    m=re.match(r'(\S+)\t월 (\d+)회',line)
    if m:sa[m.group(1)]=int(m.group(2))
S['searchad_30d']=sa
ks=[k for k in ['콜오브듀티','모던워페어4','콜오브듀티모던워페어4','모던워페어','배틀필드6'] if k in sa]
fig,ax=plt.subplots(figsize=(10,3.8));ax.barh(ks[::-1],[sa[k] for k in ks][::-1],color=[C['main'] if k=='모던워페어4' else C['light'] for k in ks][::-1])
for y,k in enumerate(ks[::-1]):ax.text(sa[k]+800,y,f'{sa[k]:,}회',va='center')
ax.set_xlim(0,max(sa.values())*1.2);ax.set_title("④ 작품 이름 '모던워페어4'가 시리즈 이름 '콜오브듀티'만큼 검색됐다\n네이버 최근 30일 검색수",loc='left')
fig.text(0.01,-0.04,'자료: 네이버 검색광고 키워드도구(띄어쓰기 없는 표기, 최근 30일 PC+모바일). 2026-10-24 조회',fontsize=8,color=C['gray'])
plt.tight_layout();plt.savefig(f'{FIG}/fig4_volume.png',bbox_inches='tight');plt.close()

from IPython.display import Image, display
display(Image(f'{FIG}/fig4_volume.png'))

In [ ]:
import json
print(json.dumps(S, ensure_ascii=False, indent=1))

## Steam 동시접속 (Step 6)
최근 콜 오브 듀티 작품은 통합 실행 앱(1938090)으로 접속이 잡혀, 작품 앱 동시접속은 0으로 나옵니다. 작품별 플레이 숫자로 쓰면 안 됩니다.

In [ ]:
st = json.load(open(f'{RAW}/steam_current_players.json'))
print(st['retrieved'])
for k, v in st['apps'].items(): print(k, v)

## 부록: 수집 코드 (기본은 실행하지 않음)

직접 다시 수집하려면 `RUN = False`를 `True`로 바꾸세요. 네이버 데이터랩 키는 Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `NAVER_ID`, `NAVER_SECRET`으로 넣습니다. 키워드도구 검색수는 네이버 검색광고 화면이나 API로 따로 확인합니다.

In [ ]:
RUN = False
import json, urllib.request
def datalab(body, cid, sec):
    req = urllib.request.Request('https://openapi.naver.com/v1/datalab/search', data=json.dumps(body).encode(), method='POST',
        headers={'X-Naver-Client-Id': cid, 'X-Naver-Client-Secret': sec, 'Content-Type': 'application/json'})
    return json.loads(urllib.request.urlopen(req).read())
def variants(name, n, short=None):
    sp = name.replace('워페어', ' 워페어').replace('옵스', ' 옵스')
    arr = [name + n, name + ' ' + n, sp + n, sp + ' ' + n, '콜오브듀티 ' + name + n, '콜오브듀티 ' + name + ' ' + n]
    if short: arr.append(short)
    return list(dict.fromkeys(arr))
T = {'MW4': variants('모던워페어', '4'), 'BO7': variants('블랙옵스', '7', '블옵7'), 'BO6': variants('블랙옵스', '6', '블옵6'),
     'MW3': variants('모던워페어', '3'), 'MW2': variants('모던워페어', '2')}
def steam_players(appid):
    u = f'https://api.steampowered.com/ISteamUserStats/GetNumberOfCurrentPlayers/v1/?appid={appid}'
    return json.loads(urllib.request.urlopen(urllib.request.Request(u, headers={'User-Agent': 'Mozilla/5.0'})).read())['response']
if RUN:
    from google.colab import userdata
    cid, sec = userdata.get('NAVER_ID'), userdata.get('NAVER_SECRET')
    w = datalab({'startDate': '2022-01-01', 'endDate': '2026-10-23', 'timeUnit': 'week',
                 'keywordGroups': [{'groupName': v[0], 'keywords': v} for v in T.values()]}, cid, sec)
    print([(r['title'], max(x['ratio'] for x in r['data'])) for r in w['results']])
    print(steam_players(1938090))